In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image
from util import visualization as u_visualization

In [ ]:
batch_size = 32

model_name = "20251021-211300.keras"
path_to_models = "../../models/finished/"

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config = load_config(f"../../logs/fit/{model_name.split('.')[0]}/config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

print(encoder_architecture)
print(classifier_architecture)



#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds*.tfrecords")

test_ds = u_dataset.get_dataset(path_to_test_data)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model

In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=True,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


In [ ]:
print(config["model"]["classifier"]["with_offsets"])
print(os.path.join(path_to_models, "encoder", f"{model_name}"))

#### Predict Data

In [ ]:
groundtruth_dataset = []
for x in test_ds:
    groundtruth_dataset.append(x)

# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))
# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True
# predictions = [model.predict(x=batch) for batch in input_list]

#### Evaluate Models

In [ ]:
results = [model.evaluate(x=batch) for batch in groundtruth_dataset]

# Reduce mean over batch axis
results_mean = tf.reduce_mean(results, axis=0)

In [ ]:
tf.print("Classifier BCE: ", results_mean[0])
tf.print("Classifier MSE: ", results_mean[2])
tf.print("Classifier Loss: ", results_mean[1])
tf.print("Encoder BCE: ", results_mean[3])
tf.print("Encoder MSE: ", results_mean[5])
tf.print("Encoder Loss: ", results_mean[4])
tf.print("Total Loss: ", results_mean[6])

In [ ]:
image, camera, intrinsics = input_list[0]
image_yuv = u_image.convert_yuyv_to_yuv(image)

#### Draw example image

In [ ]:
plt.imshow(image_yuv[0][..., 0], cmap="gray")
plt.title("Raw Example Image in Grayscale")
plt.show()

#### Show Predicted Patches


In [ ]:
object_name = "penaltyMark"

# Index: 11, 18, 23 has penaltyMark
# Index: 30 has ball
sample_index = 23

# Extract a single sample at the sample_index
sample = u_dataset.get_sample_at_index(groundtruth_dataset[0], sample_index, keep_batch=True)

#### Show Groundtruth Masks on Image


In [ ]:
image_rgb = u_image.convert_yuyv_to_rgb(sample["image"])
coordinates = u_dataset.get_coords_from_offsets(sample[object_name]["offset_mask"])

u_visualization.show_masks_on_image(
    image=image_rgb,
    coordinates=coordinates,
    mask_name=None,
    grid_dims=(15, 20),
)

u_visualization.show_masks_on_image(
    image=image_rgb,
    coordinates=coordinates,
    mask_name="object",
    grid_dims=(15, 20),
)

u_visualization.show_masks_on_image(
    image=image_rgb,
    coordinates=coordinates,
    mask_name="loss",
    grid_dims=(15, 20),
)

# offsets, _, loss= u_dataset.get_masks(coordinates=coordinates)
# print(offsets)
# print(sample[object_name]["offset_mask"])
# print(coordinates)

In [ ]:
output = model(
    (
        tf.expand_dims(image[sample_index], axis=0),
        tf.expand_dims(camera[sample_index], axis=0),
        tf.expand_dims(intrinsics[sample_index], axis=0),
    ),
    training=False,
)

model.evaluate(x=sample)

u_visualization.show_patches_on_image(image_yuv[sample_index], object_name, output["results"])
print(output["results"].keys())
tf.print(
    "Candidate 1: ",
    "Probablity: ",
    output["results"][object_name]["logits"][0][0],
    "Classification: ",
    output["results"][object_name]["classification"][0][0],
    " Masks: ",
    output["results"][object_name]["masks"][0][0],
    " Offsets (encoder): ",
    output["results"][object_name]["coords"][0][0],
)
tf.print(
    "Candidate 2: ",
    "Probablity: ",
    output["results"][object_name]["logits"][0][1],
    "Classification: ",
    output["results"][object_name]["classification"][0][1],
    " Masks: ",
    output["results"][object_name]["masks"][0][1],
    " Offsets (encoder): ",
    output["results"][object_name]["coords"][0][1],
)
tf.print(
    "Candidate 3: ",
    "Probablity: ",
    output["results"][object_name]["logits"][0][2],
    "Classification: ",
    output["results"][object_name]["classification"][0][2],
    " Masks: ",
    output["results"][object_name]["masks"][0][2],
    " Offsets (encoder): ",
    output["results"][object_name]["coords"][0][2],
)
tf.print(
    "Candidate 4: ",
    "Probablity: ",
    output["results"][object_name]["logits"][0][3],
    "Classification: ",
    output["results"][object_name]["classification"][0][3],
    " Masks: ",
    output["results"][object_name]["masks"][0][3],
    " Offsets (encoder): ",
    output["results"][object_name]["coords"][0][3],
)
tf.print(
    "Candidate 5: ",
    "Probablity: ",
    output["results"][object_name]["logits"][0][4],
    "Classification: ",
    output["results"][object_name]["classification"][0][4],
    " Masks: ",
    output["results"][object_name]["masks"][0][4],
    " Offsets (encoder): ",
    output["results"][object_name]["coords"][0][4],
)